In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("XAU_15m_data.csv", sep=";")
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

for c in ['Open','High','Low','Close']:
    df[c] = df[c].astype(float)


In [2]:
ATR_PERIOD = 14

high = df['High']
low  = df['Low']
close = df['Close']
prev_close = close.shift(1)

tr = pd.concat([
    high - low,
    (high - prev_close).abs(),
    (low - prev_close).abs()
], axis=1).max(axis=1)

df['ATR'] = tr.rolling(ATR_PERIOD).mean()


In [3]:
L, R = 3, 3
p1 = np.zeros(len(df), dtype=bool)

for i in range(L, len(df) - R):
    if df['High'][i] == df['High'][i-L:i+R+1].max():
        p1[i] = True

df['P1'] = p1


In [4]:
LOOKAHEAD = 48
MAX_ALPHA = 5.0

alpha_star = np.full(len(df), np.nan)

for i in np.where(df['P1'])[0]:
    if i + LOOKAHEAD >= len(df): 
        continue
    if np.isnan(df['ATR'][i]): 
        continue

    future_low = df['Low'][i+1:i+LOOKAHEAD+1].min()
    raw_alpha = (df['High'][i] - future_low) / df['ATR'][i]
    alpha_star[i] = min(raw_alpha, MAX_ALPHA)

df['alpha_star'] = alpha_star


upper_wick = df['High'] - df[['Open','Close']].max(axis=1)
df['Exhaustion'] = upper_wick / df['ATR']


In [5]:
test_periods = {
    "کل دیتا": None,
    "1 سال اخیر": 365,
    "6 ماه اخیر": 180,
    "3 ماه اخیر": 90,
    "1 ماه اخیر": 30,
    "2 هفته اخیر": 14,
    "1 هفته اخیر": 7,
}


In [6]:
def slice_by_days(df, days):
    if days is None:
        return df.copy()

    last_date = df['Date'].max()
    start_date = last_date - pd.Timedelta(days=days)
    return df[df['Date'] >= start_date].copy()


In [9]:
ALPHA_TRADE = 0.7
SL_ATR = 1.0
TP_ATR = 1.0
HORIZON = 48

for period_name, days in test_periods.items():

    print("-------------------")
    print("بازه:", period_name)
    print("-------------------")

    df_test = slice_by_days(df, days)

    results = []
    equity = [0]

    for i in np.where(
        (df_test['P1']) &
        (df_test['alpha_star'] >= 1.0) &
        (df_test['Exhaustion'] >= 0.6)
    )[0]:

        if i + HORIZON >= len(df_test):
            continue

        atr = df_test['ATR'].iloc[i]
        if np.isnan(atr):
            continue

        P1 = df_test['High'].iloc[i]


        entry  = P1 - ALPHA_TRADE * atr
        stop   = entry + SL_ATR * atr
        target = entry - TP_ATR * atr

        trade_open = False

        for j in range(i+1, i+HORIZON):
            hi = df_test['High'].iloc[j]
            lo = df_test['Low'].iloc[j]


            if not trade_open:
                if lo <= entry <= hi:
                    trade_open = True
            else:
                if hi >= stop:
                    results.append(-1)
                    equity.append(equity[-1] - 1)
                    break
                if lo <= target:
                    results.append(1)
                    equity.append(equity[-1] + 1)
                    break

    if len(results) == 0:
        print("!! بدون معامله")
        continue

    results = np.array(results)
    win_rate = np.mean(results == 1)

    print("تعداد معاملات:", len(results))
    print("Win Rate:", round(win_rate*100, 2), "%")


-------------------
بازه: کل دیتا
-------------------
تعداد معاملات: 12878
Win Rate: 72.22 %
-------------------
بازه: 1 سال اخیر
-------------------
تعداد معاملات: 385
Win Rate: 68.31 %
-------------------
بازه: 6 ماه اخیر
-------------------
تعداد معاملات: 138
Win Rate: 71.74 %
-------------------
بازه: 3 ماه اخیر
-------------------
تعداد معاملات: 18
Win Rate: 66.67 %
-------------------
بازه: 1 ماه اخیر
-------------------
!! بدون معامله
-------------------
بازه: 2 هفته اخیر
-------------------
!! بدون معامله
-------------------
بازه: 1 هفته اخیر
-------------------
!! بدون معامله
